    y      = E x                E  = [1, x_i, y_i]              (N x 3)
    E*     = (E^T E)^-1 E^T                                     (3 x N)
    y      = E_true alpha       E_true[i,m] = exp(i(k_m x_i + l_m y_i))
    x*     = E* E_true alpha = T alpha                          (3 x K)

    t0, tx, ty = rows of T
    H_x = tx / (i k)     R_x = |H_x|^2      -> 1 = faithfully reported
    H_y = ty / (i l)     R_y = |H_y|^2      -> 0 = invisible to the array

R is a PER-MODE ratio against the true derivative. R = 1 is good near the
origin (signal you want) and bad away from it (unresolved energy passing
straight through). The main lobe width is set by APERTURE; N controls how
well everything outside it is rejected.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# All array geometry, the model wavenumber grid, and the transfer function now
# live in array_filters.py -- the single source of truth (see that module for
# the derivation). This notebook only makes the R(k,l) summary figures.
from array_filters import (ARRAYS, LABEL, CIRC, B_KM, FX_C, FY_C,
                           transfer, half_power, metrics)

In [2]:
R = {nm: transfer(p) for nm, p in ARRAYS.items()}


In [3]:
# --------------------------------------------------------- R(k,l) figures
def comp_fig(idx, comp, fname):
    n = len(ARRAYS)
    fig, axs = plt.subplots(3, n, figsize=(3.35 * n, 11.0),
                            gridspec_kw={"height_ratios": [0.95, 1.9, 0.85]},
                            constrained_layout=True)
    for j, (nm, p) in enumerate(ARRAYS.items()):
        outer, ctr = p[:-1], p[-1]
        axl = axs[0, j]
        axl.plot(np.append(outer[:, 0], outer[0, 0]),
                 np.append(outer[:, 1], outer[0, 1]), "-", color="0.7", lw=1.0)
        axl.scatter(outer[:, 0], outer[:, 1], s=70, color="#2b6cb0", zorder=3,
                    label="outer samples")
        axl.scatter(ctr[0], ctr[1], s=70, marker="s", color="#c05621", zorder=3,
                    label="center sample")
        th = np.linspace(0, 2 * np.pi, 200)
        axl.plot(CIRC[nm]*np.cos(th), CIRC[nm]*np.sin(th), ":",
                 color="0.75", lw=0.9)
        axl.axhline(0, color="0.92", lw=0.7, zorder=0)
        axl.axvline(0, color="0.92", lw=0.7, zorder=0)
        axl.set_xlim(-18, 18); axl.set_ylim(-18, 18)
        axl.set_aspect("equal", adjustable="box")
        axl.set_xlabel("east [km]", fontsize=8)
        if j == 0:
            axl.set_ylabel("north [km]", fontsize=8)
            axl.legend(fontsize=6.5, loc="upper left", framealpha=0.9)
        axl.tick_params(labelsize=7)
        axl.set_title(f"{nm}\n({LABEL[nm]})", fontsize=9.5)

        Rm = R[nm][2 + idx]
        ax = axs[1, j]
        im = ax.pcolormesh(FX_C, FY_C, Rm, vmin=0, vmax=1,
                           cmap="viridis", shading="auto", rasterized=True)
        ax.contour(FX_C, FY_C, Rm, [0.5], colors="w", linewidths=1.4) # halfpower contour
        ax.contour(FX_C, FY_C, Rm, [0.1], colors="w", linewidths=0.6, # 10% power
                   linestyles=":")
        if Rm.max() > 1.01:
            ax.contour(FX_C, FY_C, Rm, [1.0], colors="r", linewidths=1.0)
            ax.text(0.03, 0.03, f"max R = {Rm.max():.0f}", color="r",
                    transform=ax.transAxes, fontsize=7.5, va="bottom")
        ax.set_xlabel("k [cyc km$^{-1}$]", fontsize=8)
        if j == 0:
            ax.set_ylabel("l [cyc km$^{-1}$]", fontsize=8)
        ax.set_aspect("equal")
        ax.xaxis.set_major_locator(plt.MaxNLocator(3))
        ax.yaxis.set_major_locator(plt.MaxNLocator(3))
        ax.tick_params(labelsize=7.5)
        if j == n - 1:
            fig.colorbar(im, ax=axs[1, :], shrink=0.85, extend="max",
                         label="R  (normalized)")

        axc = axs[2, j]
        ck = Rm[np.argmin(np.abs(FY_C)), :]
        cl = Rm[:, np.argmin(np.abs(FX_C))]
        axc.plot(FX_C, ck, "k", lw=1.3, label="along $k$ ($l=0$)")
        axc.plot(FY_C, cl, color="#c05621", lw=1.3, ls="--",
                 label="along $l$ ($k=0$)")
        axc.axhline(0.5, color="r", lw=0.8, ls=":")
        for f_, c_, col, dy in ((FX_C, ck, "k", 13),
                                (FY_C, cl, "#c05621", -15)):
            m = f_ > 0
            fp, cp = f_[m], c_[m]
            if (cp < 0.5).any():
                hp = fp[cp < 0.5][0]
                axc.plot(hp, 0.5, "o", ms=4.5, color=col)
                axc.annotate(f"{1/hp:.0f} km", (hp, 0.5), color=col, fontsize=7.5,
                             xytext=(5, dy), textcoords="offset points")
        axc.set_xlim(0, FX_C.max()); axc.set_ylim(0, 1.05)
        axc.set_xlabel("wavenumber [cyc km$^{-1}$]", fontsize=8)
        if j == 0:
            axc.set_ylabel("R", fontsize=8); axc.legend(fontsize=6.5)
        axc.tick_params(labelsize=7.5); axc.grid(alpha=0.25)

    d = r"\partial/\partial " + comp
    fig.suptitle(rf"Normalized power transfer $R_{{{comp}}}$ for ${d}$   "
                 r"(white solid = half power, dotted = $R$ = 0.1; "
                 r"red = $R$ = 1 where exceeded)"
                 "\n"
                 rf"equal aperture, circumradius {B_KM:.1f} km, "
                 rf"centered on 0$\degree$N 140$\degree$W",
                 fontsize=12)
    fig.savefig(fname, dpi=130)

In [4]:
comp_fig(0, "x", "fig_R_dudx.png")
comp_fig(1, "y", "fig_R_dudy.png")
